# Practical 1 - From Airfoil Geometry to Lift and Drag
## Part I: building the dataset

> **A fixed budget of 2000 aerodynamic evaluations, four inputs (m, p, t, α),
> and one question: how do you spend it?**

This morning you were asked how you would spend 5000 XFOIL evaluations across three
shape variables and an angle of attack. Now you answer it!

Nothing in this notebook trains a model. Everything here decides how good your model
is *allowed* to be.

**Roles:** Campaign designer · generator · trainer · validator · adversary.

Decide them now. The adversary's job for the next thirty minutes is to argue against
whatever the campaign designer proposes.

---
## 0 · Setup

Run this first. If anything fails, fix it before moving on.

In [ ]:
# Clone repo
%cd /content
! [ -d "/content/practical" ] && echo "Repository already cloned" || git clone https://github.com/ArnauMiro/BIP-Torino-practical.git practical
%cd /content/practical

# Colab setup. Skip locally if you already have these.
%pip install -r requirements.txt

Now let's check that everything works

In [ ]:
import sys, numpy as np
import matplotlib.pyplot as plt

import P1_ancillary as p1

print("python      ", sys.version.split()[0])
print("numpy       ", np.__version__)
import neuralfoil
print("neuralfoil  ", getattr(neuralfoil, "__version__", "unknown"))

# geometry
c = p1.naca4_coordinates(4.0, 0.4, 12.0)
assert c.shape[1] == 2 and np.allclose(c[0], [1, 0]) and np.allclose(c[-1], [1, 0])
print("geometry     ok  ", c.shape)

# one metered evaluation
_smoke = p1.NeuralFoilBudget(budget=5, tag="smoke")
out = _smoke(m=0.0, p=0.4, t=12.0, alpha=5.0)
print("neuralfoil   ok   CL=%.4f  CD=%.5f  conf=%.3f"
      % (out["CL"][0], out["CD"][0], out["analysis_confidence"][0]))
print(_smoke.report())

---
## 1 · A surrogate of a surrogate

You are about to train a neural network on the outputs of a neural network.
NeuralFoil is itself a surrogate, trained on tens of millions of XFOIL runs, which
are themselves a panel method approximation of the Navier-Stokes equations.

Two consequences you can use today:

* `model_size` gives you a **fidelity ladder inside one library**. You generate at
  `"medium"`. Your sealed test sets are labelled at the largest size. The gap between
  them is not your model's error, it is already there before you start.
* `analysis_confidence` is NeuralFoil's estimate of **its own** trustworthiness. It is
  not yours. But it is a reference to compare your `in_envelope` against.

### Warm-up — 50 free evaluations

`demo` is a **separate** evaluator with its own small budget. Spend it exploring.
It does not touch your campaign budget, and it does not roll over.

In [ ]:
demo = p1.NeuralFoilBudget(budget=50, model_size="medium", tag="demo")

# A polar for the symmetric NACA 0012, and the same for a cambered section.
alphas = np.linspace(0, 10, 11)
sym = demo(m=0.0, p=0.4, t=12.0, alpha=alphas)
cam = demo(m=4.0, p=0.4, t=12.0, alpha=alphas)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].plot(alphas, sym["CL"], "o-", label="NACA 0012")
ax[0].plot(alphas, cam["CL"], "s-", label="NACA 4412")
ax[0].set_xlabel("alpha [deg]"); ax[0].set_ylabel("CL"); ax[0].legend()

ax[1].semilogy(alphas, sym["CD"], "o-")
ax[1].semilogy(alphas, cam["CD"], "s-")
ax[1].set_xlabel("alpha [deg]"); ax[1].set_ylabel("CD  (log scale)")

p1.plot_airfoil(0.0, 0.4, 12.0, ax=ax[2])
p1.plot_airfoil(4.0, 0.4, 12.0, ax=ax[2])
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(demo.report())

**Three things to notice, and one to argue about.**

1. At α = 0 the symmetric section gives CL ≈ 0. That is a free correctness check.
   Any model of yours that fails it is broken in a way no metric will tell you about.
2. The CD axis is logarithmic. Look at what it spans, and ask what an MSE loss on raw
   CD would spend its effort on.
3. `analysis_confidence`, print it. Where is NeuralFoil already unsure?

Now argue: **is the polar above smooth because the physics is smooth, or because
NeuralFoil is C∞-continuous by construction?**

---
## 2 · The rules of the campaign

Read these before you design anything. They are not negotiable and they are not bugs.

| | |
|---|---|
| **Budget** | 2000 evaluations. Hard cap. |
| **Charged per row** | NeuralFoil is vectorised over α. Fifty angles cost fifty. |
| **Atomic rejection** | A request larger than what remains is refused *whole*. Nothing spent. |
| **No deduplication** | Ask for the same thing twice, pay twice. |
| **Low confidence costs full price** | You pay for the query, not for the answer being good. |
| **Clamp violations are free** | But they raise. They are bugs, not decisions. |

Fixed: **Re = 3 × 10⁶**, **M = 0**. Not design variables.

In [ ]:
for k, (lo, hi) in p1.CLAMP.items():
    print(f"  {k:>6s}  [{lo:6.2f}, {hi:6.2f}]")
print("\n  columns:", p1.COLUMNS)
print(f"  Re = {p1.RE:.1e}   M = {p1.MACH}")

You may sample anywhere inside the clamp. You do not have to use all of it.

Your model will be scored on **three sealed test sets**.
They are identical for every group. One sits inside the clamp; two do not.

---
## 3 · The campaign card

**Fill this in before you write a single line of sampler code.**

The `expected_strengths` and `expected_weaknesses` fields are the point. You are making
a prediction. At the end we compare predictions against leaderboards.

In [ ]:
CAMPAIGN = {
    "group":       "",              # your group name

    # --- the allocation ------------------------------------------------------
    "n_shapes":    None,            # how many distinct geometries?
    "n_alpha":     None,            # how many angles per geometry, on average?
    #  n_shapes * n_alpha must be <= 2000.  400 x 5?  40 x 50?  Something else?

    # --- where you look ------------------------------------------------------
    "bounds": {                     # your sub-box, inside bip.CLAMP
        "m":     (None, None),
        "p":     (None, None),
        "t":     (None, None),
        "alpha": (None, None),
    },
    "sampler":     "",              # "grid" | "uniform" | "lhs" | your own
    "alpha_rule":  "",              # uniform? clustered near stall? shared across shapes?

    # --- the prediction ------------------------------------------------------
    "expected_strengths":  "",      # one sentence
    "expected_weaknesses": "",      # one sentence -- be honest, it is worth more
}

assert CAMPAIGN["n_shapes"] and CAMPAIGN["n_alpha"], "fill in the allocation first"
planned = CAMPAIGN["n_shapes"] * CAMPAIGN["n_alpha"]
print(f"planned spend: {planned} of 2000   ({2000 - planned} unspent)")

### Your sampler

Write it. The signature is fixed; everything inside is yours.

```python
def sampler(n_shapes, bounds, rng) -> np.ndarray:   # -> (n_shapes, 3) of [m, p, t]
```

Things worth deciding rather than defaulting into:

* Grid or random? A grid gives you even coverage and zero corners. Random gives you
  corners and clumps.
* Do all shapes share the same α values, or does each get its own?
* The NACA 4-digit parameterisation is not a bijection. Is every geometry you generate
  actually a *different* airfoil?

In [ ]:
rng = np.random.default_rng(0)          # fix the seed. you will want to reproduce this.


def sampler(n_shapes, bounds, rng):
    '''Return an (n_shapes, 3) array of [m, p, t].'''
    raise NotImplementedError("this one is yours")


def alpha_schedule(geometries, rng):
    '''Return a list of 1-D alpha arrays, one per geometry.'''
    raise NotImplementedError("this one is yours too")

---
## 4 · Price the campaign before you buy it

`assert_valid_queries` and `check` both cost nothing. Use them. A campaign that fails
here has cost you time; a campaign that fails halfway through generation has cost you
budget you cannot get back.

In [ ]:
geoms = sampler(CAMPAIGN["n_shapes"], CAMPAIGN["bounds"], rng)
alphas_per_geom = alpha_schedule(geoms, rng)

queries = np.vstack([
    np.column_stack([np.repeat(g[None, :], a.size, axis=0), a])
    for g, a in zip(geoms, alphas_per_geom)
])

queries = p1.assert_valid_queries(queries)     # free. raises on permutation / clamp.

n_rows = queries.shape[0]
print(f"rows      {n_rows}")
print(f"budget    {n_rows} of 2000   ({2000 - n_rows} would remain)")

# coverage: look at what you are about to buy
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].scatter(queries[:, 0], queries[:, 2], s=6, alpha=.4)
ax[0].set_xlabel("m [%]"); ax[0].set_ylabel("t [%]")
ax[1].scatter(queries[:, 1], queries[:, 0], s=6, alpha=.4)
ax[1].set_xlabel("p [-]"); ax[1].set_ylabel("m [%]")
ax[2].hist(queries[:, 3], bins=30)
ax[2].set_xlabel("alpha [deg]"); ax[2].set_ylabel("count")

# draw the full clamp, so you can see how much of it you are ignoring
ax[0].set_xlim(*p1.CLAMP["m"]); ax[0].set_ylim(*p1.CLAMP["t"])
ax[1].set_xlim(*p1.CLAMP["p"]); ax[1].set_ylim(*p1.CLAMP["m"])
ax[2].set_xlim(*p1.CLAMP["alpha"])
plt.tight_layout(); plt.show()

Before you go on, three questions the adversary should be asking:

1. How many of those "distinct geometries" are distinct *airfoils*?
2. If two rows differ only in α by a fraction of a degree, and one lands in train and
   the other in validation — what will your validation error tell you?
3. What is the largest thickness you sampled? The smallest? How confident are you about
   anything outside that?

---
## 5 · Spend it

One cell. It is not reversible.

In [ ]:
nfb = p1.NeuralFoilBudget(budget=2000, model_size="medium", tag=CAMPAIGN["group"])

X, Y, conf = p1.generate_dataset(nfb, queries)

CL, CD, CM = Y[:, 0], Y[:, 1], Y[:, 2]
print(f"\nCL   [{CL.min():+.3f}, {CL.max():+.3f}]")
print(f"CD   [{CD.min():.5f}, {CD.max():.5f}]   ratio max/min = {CD.max()/CD.min():.1f}")
print(f"conf [{conf.min():.3f}, {conf.max():.3f}]   below 0.5: {np.sum(conf < 0.5)} rows")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].hist(CL, bins=40); ax[0].set_xlabel("CL")
ax[1].hist(CD, bins=40); ax[1].set_xlabel("CD"); ax[1].set_yscale("log")
ax[2].scatter(X[:, 3], conf, s=6, alpha=.3)
ax[2].set_xlabel("alpha [deg]"); ax[2].set_ylabel("analysis_confidence")
plt.tight_layout(); plt.show()

Look hard at the middle panel before moving on.

The `AeroSurrogate` asks your model to return **`log10(CD)`**, not `CD`.
That is not an arbitrary formatting choice. Work out why, and note that nothing in the
dataset does the transform for you.

---
## 6 · Export

Two files, and the JSON matters as much as the arrays. It carries your campaign card and
the full spend log: every call, every α range, every refusal. On Wednesday, when your
agent does something absurd, this is the record you will be reading to work out why.

In [ ]:
p1.save_campaign("campaign", X, Y, conf, nfb, campaign_card=CAMPAIGN)

X2, Y2, conf2, meta = p1.load_campaign("campaign")
assert np.array_equal(X, X2) and np.array_equal(Y, Y2)
print("saved and verified:  campaign.npz  +  campaign.json")
print(f"  {meta['n_rows']} rows, spent {meta['spent']}/{meta['budget']}")

---

**You now have a dataset and no model.** Part II trains one.

Whatever happens next is bounded by what you just did. Keep the campaign card open.
At the end you will be asked to explain the gap between what you predicted and what the leaderboards say.